# 09 — End-to-End Dark-Web SOC Pipeline

Everything we've built, wired together. This notebook takes **raw dark-web text** (CoDA pages + seeded injections + leak-site posts + credential dumps) and produces a **ranked, deduplicated alert feed** — the output a SOC analyst would actually triage.

## The pipeline

```
raw input
   │
   ├── onion page  ─► [nb 04] category classifier  ──► route to leak-site branch if Hacking/Financial
   │                        │
   │                        ▼
   │                  [nb 07] leak-site binary classifier ─► flagged posts extract victim
   │                        │
   │                        ▼
   │                  [nb 05] entity extractor        ─► CVEs, wallets, emails, actors, orgs
   │                        │
   │                        ▼
   │                  [nb 06] asset matcher           ─► customer-specific alerts
   │
   └── credential dump ─► [nb 08] parser + matcher    ─► employee-credential alerts
                              │
                              ▼
                       HIBP enrichment
                              │
                              ▼
                    ── unified alert feed ──
                     (ranked, deduped, JSON)
```

## Prerequisites

- **Nb 04** (DarkBERT CoDA classifier) — saved at `models/darkbert_coda_final`
- **Nb 05** (entity extractor) — saved at `models/darkbert_entity_final`
- **Nb 07** (leak-site classifier) — saved at `models/darkbert_leaksite_final`
- **Nb 08** — synthetic credential dumps in `data/synthetic_dumps/`

## 1 — Load all models

In [ ]:
import json, re, hashlib
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd, numpy as np
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForTokenClassification)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_seq_cls(path, has_threshold=False):
    path = Path(path)
    tk = AutoTokenizer.from_pretrained(str(path), use_fast=True)
    m = AutoModelForSequenceClassification.from_pretrained(str(path)).to(device).eval()
    threshold = 0.5
    if has_threshold and (path / 'threshold.json').exists():
        with open(path / 'threshold.json') as f:
            threshold = json.load(f)['threshold']
    labels = [m.config.id2label[i] for i in range(m.config.num_labels)]
    return tk, m, labels, threshold

def load_tok_cls(path):
    path = Path(path)
    tk = AutoTokenizer.from_pretrained(str(path), use_fast=True)
    m = AutoModelForTokenClassification.from_pretrained(str(path)).to(device).eval()
    with open(path / 'tag_vocab.json') as f:
        tag_vocab = json.load(f)
    id2tag = {i: t for i, t in enumerate(tag_vocab['tags'])}
    return tk, m, id2tag

print('Loading category classifier (nb 04)...')
cat_tk, cat_m, cat_labels, _ = load_seq_cls('models/darkbert_coda_final')
print(f'  categories: {cat_labels}')

print('Loading entity extractor (nb 05)...')
ent_tk, ent_m, ent_id2tag = load_tok_cls('models/darkbert_entity_final')

print('Loading leak-site classifier (nb 07)...')
leak_tk, leak_m, leak_labels, leak_threshold = load_seq_cls('models/darkbert_leaksite_final',
                                                              has_threshold=True)
print(f'  threshold: {leak_threshold:.2f}')
print('\nAll models loaded on', device)

## 2 — The pipeline functions

Thin wrappers around each model, all returning plain Python dicts. Composable.

In [ ]:
MAX_LEN = 512
STRIDE  = 64

def classify_category(text):
    enc = cat_tk(text, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.softmax(cat_m(**enc).logits[0], dim=-1).cpu().numpy()
    i = int(probs.argmax())
    return {'category': cat_labels[i], 'confidence': float(probs[i])}

def classify_leak(text):
    enc = leak_tk(text, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.softmax(leak_m(**enc).logits[0], dim=-1).cpu().numpy()
    p_leak = float(probs[1])
    return {'is_leak': p_leak >= leak_threshold, 'leak_score': p_leak}

def extract_entities(text):
    enc = ent_tk(text, truncation=True, max_length=MAX_LEN,
                  return_offsets_mapping=True, return_overflowing_tokens=True,
                  stride=STRIDE, return_tensors='pt', padding=True).to(device)
    offsets = enc.pop('offset_mapping').cpu().numpy()
    enc.pop('overflow_to_sample_mapping', None)
    with torch.no_grad():
        tags = ent_m(**enc).logits.argmax(-1).cpu().numpy()
    seen = {}
    for chunk_tags, chunk_offs in zip(tags, offsets):
        cur = None
        for (s, e), tid in zip(chunk_offs, chunk_tags):
            if s == e == 0: continue
            tag = ent_id2tag[int(tid)]
            if tag == 'O':
                if cur: seen[(cur[0], cur[1])] = cur; cur = None
            elif tag.startswith('B-'):
                if cur: seen[(cur[0], cur[1])] = cur
                cur = [int(s), int(e), tag[2:]]
            elif tag.startswith('I-') and cur and cur[2] == tag[2:]:
                cur[1] = int(e)
            else:
                if cur: seen[(cur[0], cur[1])] = cur
                cur = [int(s), int(e), tag[2:]]
        if cur: seen[(cur[0], cur[1])] = cur
    return [{'start': s, 'end': e, 'label': l, 'text': text[s:e]} for (s, e), (_, _, l) in seen.items()]

## 3 — Customer profile (reuse from nb 06)

In [ ]:
CUSTOMER = {
    'name':      'Acme Industrial Corp',
    'aliases':   ['Acme Industrial', 'AcmeCorp', 'Acme Corp'],
    'domains':   ['acme-industrial.com', 'acmeind.com', 'acme-corp.net'],
    'brands':    ['AcmePlant', 'AcmeOps', 'AcmeCloud'],
    'execs':     ['Alice Brennan', 'Vikram Sahni', 'Marcus Chen'],
    'known_ips': ['203.0.113.42', '198.51.100.17'],
}

from difflib import SequenceMatcher
def norm(s): return re.sub(r'\s+', '', s.strip().lower())
def fuzzy(a, b): return SequenceMatcher(None, norm(a), norm(b)).ratio()

cust_doms = {d.lower() for d in CUSTOMER['domains']}
cust_ips  = set(CUSTOMER['known_ips'])
cust_terms = {b.lower() for b in CUSTOMER['brands'] + CUSTOMER['aliases'] + CUSTOMER['execs']} | {CUSTOMER['name'].lower()}

def asset_match(entities, text):
    hits = []
    text_lo = text.lower()
    for e in entities:
        surf_lo = e['text'].lower()
        if e['label'] == 'EMAIL' and '@' in surf_lo:
            if surf_lo.split('@')[-1] in cust_doms:
                hits.append({'type': 'employee_email', 'surface': e['text'], 'span': (e['start'], e['end'])})
        elif e['label'] == 'IP_ADDRESS' and e['text'] in cust_ips:
            hits.append({'type': 'infrastructure_ip', 'surface': e['text'], 'span': (e['start'], e['end'])})
        elif e['label'] == 'ORGANIZATION':
            r = max(fuzzy(e['text'], x) for x in [CUSTOMER['name']] + CUSTOMER['aliases'])
            if r >= 0.85:
                hits.append({'type': 'org_mention', 'surface': e['text'], 'span': (e['start'], e['end']), 'score': round(r, 3)})
    for dom in cust_doms:
        for m in re.finditer(re.escape(dom), text_lo):
            hits.append({'type': 'domain_mention', 'surface': text[m.start():m.end()], 'span': (m.start(), m.end())})
    for term in cust_terms:
        for m in re.finditer(r'\b' + re.escape(term) + r'\b', text_lo):
            hits.append({'type': 'brand_mention', 'surface': text[m.start():m.end()], 'span': (m.start(), m.end())})
    return hits

## 4 — Full pipeline: onion-page path

Reads one page text → category → (if relevant) leak-site check → entities → asset matches → unified alert record.

In [ ]:
SEV_WEIGHT = {
    'employee_email': 10, 'infrastructure_ip': 9,
    'org_mention': 7, 'domain_mention': 5, 'brand_mention': 3,
}
CAT_MULT = {'Hacking': 1.5, 'Financial': 1.4, 'Crypto': 1.2}

def snippet(text, span, window=120):
    s = max(0, span[0] - window); e = min(len(text), span[1] + window)
    return ('…' if s > 0 else '') + text[s:e].replace('\n', ' ') + ('…' if e < len(text) else '')

def severity_band(score):
    if score >= 12: return 'critical'
    if score >= 8:  return 'high'
    if score >= 5:  return 'medium'
    return 'low'

def process_onion_page(page_id, text):
    cat_info = classify_category(text)
    is_ransom_candidate = cat_info['category'] in {'Hacking', 'Financial'}
    leak_info = classify_leak(text) if is_ransom_candidate else {'is_leak': False, 'leak_score': 0.0}
    entities = extract_entities(text)
    hits = asset_match(entities, text)

    # Any signal worth emitting?
    emit = bool(hits) or leak_info['is_leak']
    if not emit:
        return None

    if hits:
        base = max(SEV_WEIGHT.get(h['type'], 1) for h in hits)
        score = base * CAT_MULT.get(cat_info['category'], 1.0) + (len({h['type'] for h in hits}) - 1) * 2
    else:
        score = 6.0  # leak post without customer match — still notable
    if leak_info['is_leak']:
        score += 4
    score = round(score, 2)

    headline_hit = max(hits, key=lambda h: SEV_WEIGHT.get(h['type'], 0)) if hits else None
    victim_orgs = [e['text'] for e in entities if e['label'] == 'ORGANIZATION'][:5]

    return {
        'alert_id':    hashlib.sha256(f'onion-{page_id}'.encode()).hexdigest()[:12],
        'source_type': 'onion_page',
        'source_id':   page_id,
        'customer':    CUSTOMER['name'],
        'severity':    severity_band(score),
        'severity_score': score,
        'category':    cat_info['category'],
        'category_confidence': round(cat_info['confidence'], 3),
        'is_ransomware_leak': leak_info['is_leak'],
        'leak_score':  round(leak_info['leak_score'], 3),
        'match_types': sorted({h['type'] for h in hits}),
        'headline':    f'{headline_hit["type"]}: {headline_hit["surface"]}' if headline_hit else (
                        f'ransomware leak post — victims: {victim_orgs}' if leak_info['is_leak'] else 'entity-only match'),
        'evidence':    snippet(text, headline_hit['span']) if headline_hit else text[:300],
        'extracted_entities': [
            {'label': e['label'], 'text': e['text']}
            for e in entities
            if e['label'] in {'CVE', 'CRYPTO_WALLET', 'EMAIL', 'IP_ADDRESS', 'MALWARE_FAMILY', 'THREAT_ACTOR', 'ORGANIZATION'}
        ][:20],
    }

## 5 — Credential-dump path (from nb 08)

In [ ]:
from urllib.parse import urlparse

EMAIL_RE = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')
def is_email(s): return bool(s and EMAIL_RE.match(s))
def url_to_domain(u):
    if not u: return None
    try:
        return urlparse(u if '://' in u else 'http://' + u).netloc.lower() or None
    except Exception:
        return None

def iter_combo(path):
    for line_no, line in enumerate(Path(path).read_text().splitlines(), 1):
        if ':' in line:
            u, p = line.split(':', 1)
            yield {'email': u if is_email(u) else None, 'password': p, 'domain': u.split('@')[-1].lower() if is_email(u) else None,
                   'source': f'{path}:{line_no}', 'format': 'combo'}

def iter_stealer(path):
    for line_no, line in enumerate(Path(path).read_text().splitlines(), 1):
        parts = line.split(';')
        if len(parts) >= 3:
            url, user, pw = parts[0], parts[1], ';'.join(parts[2:])
            email = user if is_email(user) else None
            yield {'email': email, 'password': pw,
                   'domain': url_to_domain(url) or (email.split('@')[-1].lower() if email else None),
                   'source': f'{path}:{line_no}', 'format': 'stealer'}

def iter_csv(path):
    df = pd.read_csv(path)
    for i, row in df.iterrows():
        email = row.get('email') if is_email(row.get('email')) else None
        yield {'email': email, 'password': row.get('password_hash') or row.get('password'),
               'domain': url_to_domain(row.get('source_url')) or (email.split('@')[-1].lower() if email else None),
               'source': f'{path}:{i+2}', 'format': 'csv'}

def process_credential_dumps(dump_dir='data/synthetic_dumps'):
    d = Path(dump_dir)
    if not d.exists():
        return []
    all_recs = []
    for p in d.glob('combo_*.txt'):      all_recs.extend(iter_combo(p))
    for p in d.glob('*stealer*.txt'):    all_recs.extend(iter_stealer(p))
    for p in d.glob('redline_logs*.txt'):all_recs.extend(iter_stealer(p))
    for p in d.glob('*.csv'):            all_recs.extend(iter_csv(p))

    alerts = []
    for r in all_recs:
        dom = r.get('domain') or ''
        if dom in cust_doms:
            pw = r.get('password') or ''
            alerts.append({
                'alert_id': hashlib.sha256(f'cred-{r["email"]}{r["password"]}{r["source"]}'.encode()).hexdigest()[:12],
                'source_type': 'credential_dump',
                'source_id':  r['source'],
                'customer':   CUSTOMER['name'],
                'severity':   'critical',
                'severity_score': 15.0,
                'category':   None,
                'match_types': ['employee_credential'],
                'headline':   f'employee_credential exposed: {r["email"]}',
                'evidence':   f'Format {r["format"]}; password preview {pw[:2] + "***" + pw[-1] if pw else None}',
                'extracted_entities': [{'label': 'EMAIL', 'text': r['email']}],
            })
    return alerts

## 6 — Run the pipeline end-to-end on a mixed input

Combine:
- the **seeded CoDA corpus from nb 06** (has known injected hits)
- the **synthetic credential dumps from nb 08**

Run both branches. Merge. Rank. Dedupe.

In [ ]:
# --- Load the same seeded corpus as nb 06 -----------------------------
CODA_DIR = Path('data/coda/coda_dataset')
rows = []
for f in CODA_DIR.glob('*.txt'):
    parts = f.stem.split('-')
    if len(parts) < 4 or parts[2] != 'en' or parts[1] == 'Others': continue
    text = f.read_text(errors='ignore').strip()
    if len(text.split()) < 20: continue
    rows.append({'id': parts[0], 'category': parts[1], 'text': text})
coda = pd.DataFrame(rows).reset_index(drop=True)

# Inject the same 40 hits as nb 06 (seeded by same random_state)
import random
random.seed(7)
RECIPES = [
    ('Financial', 'Fresh Acme Industrial Corp email dumps: john.smith@acme-industrial.com with working password.', 10),
    ('Hacking',   'Selling RDP access to acme-industrial.com — admin panel, MFA bypassed. Price: 0.3 BTC.', 8),
    ('Hacking',   '*** LEAKED *** Acme Industrial Corp database — 240GB including customer records and HR data.', 5),
    ('Crypto',    'Executive targeting: Alice Brennan from Acme Industrial Corp. LinkedIn profile scraped.', 4),
    ('Financial', 'Stealer log hit: sarah.j@acmeind.com:Password123! from acme-industrial.com Outlook login.', 8),
    ('Hacking',   'New phish kit impersonates acme-industrial.com. Looking for customers, $300 setup.', 5),
]
for cat, tmpl, n in RECIPES:
    pool = coda[coda['category'] == cat].sample(n, random_state=random.randint(0, 1000)).index.tolist()
    for idx in pool:
        coda.at[idx, 'text'] = tmpl + ' ' + coda.at[idx, 'text']

# --- Sample for feasible runtime — the pipeline is heavy ---
# Always include Hacking/Financial/Crypto since seeded hits live there; fill with random others.
priority = coda[coda['category'].isin(['Hacking', 'Financial', 'Crypto'])]
other    = coda[~coda['category'].isin(['Hacking', 'Financial', 'Crypto'])].sample(300, random_state=42)
work_set = pd.concat([priority, other]).sample(frac=1, random_state=0).reset_index(drop=True)
print(f'Running pipeline over {len(work_set)} pages (all priority categories + 300 sampled others)...')

onion_alerts = []
for i, row in work_set.iterrows():
    a = process_onion_page(row['id'], row['text'])
    if a is not None:
        onion_alerts.append(a)
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(work_set)}  alerts so far: {len(onion_alerts)}')

print(f'\nOnion-page alerts: {len(onion_alerts)}')
cred_alerts = process_credential_dumps()
print(f'Credential-dump alerts: {len(cred_alerts)}')

## 7 — Merge, dedupe, rank

In [ ]:
all_alerts = onion_alerts + cred_alerts

# Dedupe: if an email appears in both an onion page alert AND a credential dump alert,
# keep the credential dump (higher confidence) — real implementations dedupe on an
# indicator hash (email|domain|artifact).
def indicator(a):
    ents = a.get('extracted_entities', [])
    emails = tuple(sorted(e['text'].lower() for e in ents if e['label'] == 'EMAIL'))
    return emails or (a['source_id'],)

seen = {}
SOURCE_PRIORITY = {'credential_dump': 0, 'onion_page': 1}
for a in all_alerts:
    key = indicator(a)
    if key not in seen or SOURCE_PRIORITY[a['source_type']] < SOURCE_PRIORITY[seen[key]['source_type']]:
        seen[key] = a
unified = sorted(seen.values(), key=lambda a: -a['severity_score'])

print(f'Unified alert feed: {len(unified)} alerts (from {len(all_alerts)} pre-dedup)')
print('\nSeverity distribution:')
print(pd.Series([a["severity"] for a in unified]).value_counts().to_string())
print('\nSource-type distribution:')
print(pd.Series([a["source_type"] for a in unified]).value_counts().to_string())

## 8 — The analyst view

What a SOC analyst sees when they open their dark-web-monitoring queue in the morning.

In [ ]:
print('╔' + '═'*76 + '╗')
print(f'║ {"DARK WEB MONITORING — DAILY ALERT FEED":^76s} ║')
print(f'║ {"Customer: " + CUSTOMER["name"]:^76s} ║')
print('╚' + '═'*76 + '╝\n')

for severity in ['critical', 'high', 'medium']:
    group = [a for a in unified if a['severity'] == severity]
    if not group: continue
    print(f'── {severity.upper()} ({len(group)} alert{"s" if len(group)!=1 else ""}) ' + '─' * (60 - len(severity)))
    for a in group[:8]:  # show first 8 per band
        src = f'[{a["source_type"]}]'
        print(f'{src:<18s} score={a["severity_score"]:>5}  {a["alert_id"]}')
        print(f'                   {a["headline"]}')
        ev = a['evidence'].replace('\n', ' ')[:200]
        print(f'                   └─ {ev}')
        print()
    if len(group) > 8:
        print(f'                   ... and {len(group)-8} more {severity} alerts')
        print()

## 9 — Save the unified feed

JSONL is the lingua franca of SIEMs (Splunk, Elastic, Sentinel all ingest it). One JSON object per line; each line is one alert.

In [ ]:
OUT = Path('processed/unified_alerts.jsonl')
with open(OUT, 'w') as f:
    for a in unified:
        f.write(json.dumps(a) + '\n')
print(f'Wrote {len(unified)} alerts to {OUT}')

print('\nExample alert (rendered):')
print(json.dumps(unified[0], indent=2))

## What you've built

A complete dark-web CTI pipeline, end-to-end:

- **Ingest** — mixed inputs: onion pages (CoDA + seeded hits) and credential dumps (synthetic).
- **Classify** — DarkBERT CoDA categories from nb 04, DarkBERT leak-site binary from nb 07.
- **Extract** — DarkBERT token classifier from nb 05 surfaces CVEs, wallets, emails, threat actors, malware, victim orgs.
- **Match** — multi-tier matcher from nb 06 checks entities + raw text against a synthetic customer profile.
- **Parse** — four credential-dump formats from nb 08, all feeding the same alert schema.
- **Unify** — ranked, deduplicated, severity-banded JSONL feed ready for SIEM ingestion.

## What this is and isn't

**This is** a faithful demonstration of how a real dark-web-monitoring product composes models and rules to produce analyst-ready intelligence. The shape of the code, the alert schema, the severity logic, the dedup strategy — all of these directly transfer to a production version.

**This isn't** production-ready. Missing pieces for real deployment:
- **Collection.** We start with labeled research data (CoDA) and synthetic dumps. Real systems have Tor-scraping infrastructure, forum-mirror rotators, Telegram ingesters, marketplace-API scrapers.
- **Scale.** We process pages in-memory one-at-a-time. Real systems run this across Kubernetes with message queues.
- **Indicator storage / historical search.** Alerts here are one-shot JSON. Real systems write to a SIEM + a searchable indicator database, with MTTR tracking.
- **Feedback loops.** Analyst false-positive labels should flow back into model retraining. That's an MLOps layer we don't build here.
- **Ethics + legal.** Scraping Tor, handling real stolen credentials, and operating a commercial dark-web monitor all involve legal questions specific to jurisdiction and contract. Handled as a separate discipline, not code.

## What's next

The CTI 301 series has walked the path from *"tune BERT on CTI classification"* (nb 01) to *"ship an end-to-end dark-web SOC pipeline"* (this notebook). Natural next directions:

- **CTI 401 — Collection & ingestion.** Tor scraping, forum mirroring, Telegram/Discord CTI channels, API-based IOC feeds. Where the data *actually* comes from.
- **CTI 402 — RAG over a threat-intel knowledge base.** The Claude-API version of this pipeline — instead of "extract entities and match," an analyst asks natural-language questions and the system retrieves + summarizes.
- **CTI 403 — Actor tracking over time.** Temporal modeling of threat actor behavior, TTPs drift, campaign clustering across incidents.